# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Track 4 · Ngày 1 · VinUniversity AICB 2026**  
**Họ và tên:** Nguyễn Thế Khang  
**Mã số sinh viên (MSSV):** 2A202602964  

> **Câu hỏi dẫn dắt của bài học:** *"Một mạng có loss không giảm sau 2 000 bước huấn luyện. Lỗi nằm ở dữ liệu, ở kiến trúc, hay ở vòng lặp huấn luyện?"*  
> Notebook này triển khai hoàn chỉnh pipeline huấn luyện, kiểm tra sức khoẻ ban đầu, đo độ nhiễu giữa các seed, thực hiện các nhóm thí nghiệm (Hàm mất mát, Bộ tối ưu, Hyper-parameter, Dropout, Gradient Clipping, Mixed Precision, Khởi tạo tham số) và đánh giá trên tập eval độc lập.

## 0. Thiết lập môi trường và cấu hình đường dẫn

In [1]:
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Cấu hình đường dẫn
REPO_ROOT = "../.." if os.path.exists("../../data") else ".."
OUT_DIR = ".." if os.path.exists("../figures") else "."

# Thiết bị tính toán
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version : {torch.__version__}")
print(f"Thiết bị (Device): {device}")
if torch.cuda.is_available():
    print(f"Tên GPU         : {torch.cuda.get_device_name(0)}")

# Tạo các thư mục lưu trữ kết quả nếu chưa có
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

# Import các module từ code/
from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
print("Đã import thành công toàn bộ thư viện và module nội bộ!")

PyTorch Version : 2.14.0+cpu
Thiết bị (Device): cpu


Đã import thành công toàn bộ thư viện và module nội bộ!


## Part 0 — Chuẩn bị dữ liệu và phân tách Validation

- Dữ liệu Forest CoverType gồm 581,012 mẫu, 54 đặc trưng (10 cột số liên tục và 44 cột one-hot).
- Phân tách cố định: 464,809 train / 116,203 eval bằng `split_metadata.csv`.
- Tách 20% validation từ tập train (phân tầng theo nhãn, seed 42) -> 371,847 train / 92,962 val.
- Chuẩn hoá (Z-score) **chỉ tính mean/std trên 10 cột số của tập train còn lại** để tránh rò rỉ thông tin (data leakage) sang val và eval.

In [2]:
# Đảm bảo đã có dữ liệu xử lý sơ bộ
processed_dir = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{processed_dir}/train.npz"):
    print("Chạy script split_data.py...")
    subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/split_data.py"], cwd=REPO_ROOT, check=True)

# Tải dữ liệu, phân tầng val 20%, chuẩn hoá 10 cột liên tục và đưa lên tensor
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

print("\n--- Kiểm tra phân bố và tính chất dữ liệu ---")
print(f"Train size: {len(data['X_tr']):,d} mẫu | Val size: {len(data['X_val']):,d} mẫu | Eval size: {len(data['X_eval']):,d} mẫu")
print(f"Mốc đoán lớp đa số trên Val (Class {data['majority_class']}): {data['majority_acc']:.4f}")
assert abs(data['majority_acc'] - 0.4876) < 0.005, "Accuracy đoán lớp đa số phải xấp xỉ 0.4876!"

Dataset summary:
  Train : 371,847 mẫu (80%)
  Val   : 92,962 mẫu (20%)
  Eval  : 116,203 mẫu
  Majority class on Val: class 1 (acc = 0.4876)
  Standardized train features: mean max=3.21e-04, std mean=1.0000

--- Kiểm tra phân bố và tính chất dữ liệu ---
Train size: 371,847 mẫu | Val size: 92,962 mẫu | Eval size: 116,203 mẫu
Mốc đoán lớp đa số trên Val (Class 1): 0.4876


## Part 1 — Định nghĩa Model và kiểm tra "sức khoẻ" ban đầu

Theo quy định kiến trúc:
- `M-base`: $54 \to 256 \to 128 \to 7$, đúng **47,879 tham số**.
- Đầu ra là logits thô $(B, 7)$, không có softmax trong model.
- Dropout chỉ đặt sau ReLU của lớp ẩn.

**3 phép thử sức khoẻ bắt buộc (Karpathy's Recipe & Slide Chương 5):**
1. Kiểm tra tham số và shape: $B \times 54 \to B \times 7$.
2. Loss bước 0: Với model vừa khởi tạo He, loss Cross-Entropy trên val phải xấp xỉ $\ln(7) \approx 1.946$.
3. Quá khớp 20 mẫu: Huấn luyện một lô nhỏ 20 mẫu tắt dropout, loss phải hạ về xấp xỉ 0 (accuracy 100%).
4. Kiểm tra dòng chảy gradient: Sau một lần backward, mọi tham số phải có gradient khác None và khác 0.

In [3]:
set_seed(42)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)

# 1. Assert số lượng tham số
params_count = count_params(model)
print(f"1. Số lượng tham số M-base: {params_count:,d} (Kỳ vọng: {EXPECTED_PARAMS[(256, 128)]:,d})")
assert params_count == EXPECTED_PARAMS[(256, 128)], "Sai lệch số lượng tham số!"

# 2. Kiểm tra shape logits
dummy_x = torch.randn(8, 54, device=device)
dummy_logits = model(dummy_x)
print(f"2. Shape logits đầu ra cho batch (8, 54): {dummy_logits.shape}")
assert dummy_logits.shape == (8, 7), "Shape logits không đúng!"

# 3. Loss bước 0 trên tập val
step0 = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
print(f"3. Loss bước 0 trên val: {step0['loss']:.4f} (ln 7 = 1.9459, độ lệch {step0['loss'] - 1.9459:+.4f})")

# 4. Quá khớp 20 mẫu
x20, y20 = data["X_tr"][:20], data["y_tr"][:20]
overfit_m = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt_overfit = torch.optim.Adam(overfit_m.parameters(), lr=0.02)
losses_20 = []
for step in range(120):
    opt_overfit.zero_grad()
    l = F.cross_entropy(overfit_m(x20), y20)
    l.backward()
    opt_overfit.step()
    losses_20.append(l.item())
print(f"4. Quá khớp 20 mẫu: loss tại bước 120 = {losses_20[-1]:.6f} (loss ban đầu: {losses_20[0]:.4f})")
assert losses_20[-1] < 0.05, "Không overfit được 20 mẫu!"

# 5. Kiểm tra gradient chảy tới mọi tham số
opt_overfit.zero_grad()
l = F.cross_entropy(overfit_m(x20), y20)
l.backward()
for name, p in overfit_m.named_parameters():
    assert p.grad is not None and p.grad.norm().item() > 0, f"Gradient đứt gãy tại {name}!"
print("5. Gradient chảy thông suốt tới 100% các lớp và tham số.")
print("==> MÔ HÌNH VƯỢT QUA TOÀN BỘ CÁC BÀI TEST SỨC KHOẺ BAN ĐẦU!")

1. Số lượng tham số M-base: 47,879 (Kỳ vọng: 47,879)
2. Shape logits đầu ra cho batch (8, 54): torch.Size([8, 7])
3. Loss bước 0 trên val: 2.3776 (ln 7 = 1.9459, độ lệch +0.4317)


4. Quá khớp 20 mẫu: loss tại bước 120 = 0.000006 (loss ban đầu: 1.9089)
5. Gradient chảy thông suốt tới 100% các lớp và tham số.
==> MÔ HÌNH VƯỢT QUA TOÀN BỘ CÁC BÀI TEST SỨC KHOẺ BAN ĐẦU!


## Part 2 — Pipeline huấn luyện và Đo độ nhiễu Baseline

Hàm `run_experiment(cfg, data)` được đóng gói duy nhất, nhận dict cấu hình và thực thi huấn luyện.  
Để so sánh công bằng và có căn cứ khoa học, mô hình baseline được chạy qua **3 seed khác nhau** (Seed 1, 2, 3) để xác định độ lệch chuẩn $\sigma$ và **ngưỡng nhiễu $2\sigma$**.  
Bất kỳ sự cải thiện nào nhỏ hơn $2\sigma$ đều không được coi là bằng chứng xác thực.

In [4]:
# Baseline cấu hình: SGD+Momentum 0.9, He init, batch 512, 20 epochs, lr=0.05
base_results = []
for s in (1, 2, 3):
    cfg_s = {**DEFAULT_CFG, "lr": 0.05, "seed": s, "exp_id": f"base-s{s}", "description": f"Baseline M-base (seed {s})"}
    # Kiểm tra nếu đã chạy trước đó thì nạp lại, nếu chưa thì chạy
    json_path = f"{OUT_DIR}/results/{cfg_s['exp_id']}.json"
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            res = json.load(f)
    else:
        res = run_experiment(cfg_s, data)
        save_result(res, f"{OUT_DIR}/results")
        plot_run(res, f"{OUT_DIR}/figures/{cfg_s['exp_id']}.png")
    base_results.append(res)

# Tính trung bình và độ lệch chuẩn
f1_scores = [r["summary"]["val_macro_f1"] for r in base_results]
acc_scores = [r["summary"]["val_acc"] for r in base_results]
mean_f1, std_f1 = np.mean(f1_scores), np.std(f1_scores, ddof=1)
mean_acc, std_acc = np.mean(acc_scores), np.std(acc_scores, ddof=1)
noise_threshold_2sigma = 2 * std_f1

print(f"Baseline Val Macro-F1: {mean_f1:.4f} ± {std_f1:.4f} (Các seed: {[round(x, 4) for x in f1_scores]})")
print(f"Baseline Val Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"==> NGƯỠNG NHIỄU (2σ của Val Macro-F1): {noise_threshold_2sigma:.4f}")

Baseline Val Macro-F1: 0.8387 ± 0.0015 (Các seed: [0.8405, 0.8376, 0.8382])
Baseline Val Accuracy: 0.8986 ± 0.0006
==> NGƯỠNG NHIỄU (2σ của Val Macro-F1): 0.0030


## Part 3 — Thí nghiệm hệ thống trên 7 Chủ đề

Mỗi thí nghiệm tuân thủ chặt chẽ:  
1. **Dự đoán trước** khi chạy (nguyên lý lý thuyết).  
2. **Chỉ đổi MỘT yếu tố** duy nhất so với baseline.  
3. **Đối chiếu kết quả** với baseline và kiểm tra vượt ngưỡng nhiễu $2\sigma$.

In [5]:
# Danh sách kết quả tổng hợp
all_results = load_results(f"{OUT_DIR}/results")
results_by_id = {r["cfg"]["exp_id"]: r for r in all_results}
print(f"Số lượng thí nghiệm đã hoàn tất trong hệ thống: {len(results_by_id)}")
for exp_id, r in results_by_id.items():
    print(f" - {exp_id:20s}: Val Acc = {r['summary']['val_acc']:.4f}, Val Macro-F1 = {r['summary']['val_macro_f1']:.4f}")

Số lượng thí nghiệm đã hoàn tất trong hệ thống: 26
 - amp-bf16            : Val Acc = 0.8988, Val Macro-F1 = 0.8405
 - base-s1             : Val Acc = 0.8988, Val Macro-F1 = 0.8405
 - base-s2             : Val Acc = 0.8991, Val Macro-F1 = 0.8376
 - base-s3             : Val Acc = 0.8978, Val Macro-F1 = 0.8382
 - clip-highlr-clip1.0 : Val Acc = 0.8905, Val Macro-F1 = 0.8210
 - clip-highlr-noclip  : Val Acc = 0.8810, Val Macro-F1 = 0.8000
 - clip-norm-1.0       : Val Acc = 0.8994, Val Macro-F1 = 0.8348
 - drop-0.1            : Val Acc = 0.8924, Val Macro-F1 = 0.8233
 - drop-0.3            : Val Acc = 0.8600, Val Macro-F1 = 0.7558
 - final-model         : Val Acc = 0.9181, Val Macro-F1 = 0.8759
 - hp-batch-128        : Val Acc = 0.9123, Val Macro-F1 = 0.8630
 - hp-batch-2048       : Val Acc = 0.8742, Val Macro-F1 = 0.7804
 - hp-m-deep           : Val Acc = 0.9151, Val Macro-F1 = 0.8659
 - hp-m-wide           : Val Acc = 0.9087, Val Macro-F1 = 0.8615
 - init-normal         : Val Acc = 0.88

### 3.1 Nhóm Hàm mất mát (Loss): Cross-Entropy vs MSE

- **Dự đoán:** Cross-Entropy (CE) sẽ vượt trội MSE vì gradient của CE tỉ lệ thuận với sai số $(p - y)$, không bị bão hoà khi dự đoán sai lệch lớn. Ngược lại MSE với nhãn one-hot sinh gradient nhỏ khi hàm kích hoạt ở vùng bão hoà, khiến mô hình học chậm hơn.

In [6]:
loss_exps = [results_by_id["base-s1"], results_by_id["loss-mse"]]
for r in loss_exps:
    print(f"{r['cfg']['exp_id']:12s} ({r['cfg']['loss'].upper()}): Val Acc = {r['summary']['val_acc']:.4f}, Val Macro-F1 = {r['summary']['val_macro_f1']:.4f}")
plot_compare(loss_exps, "val_macro_f1", f"{OUT_DIR}/figures/compare_loss.png", title="So sánh Hàm mất mát: CE vs MSE (Val Macro-F1)")

base-s1      (CE): Val Acc = 0.8988, Val Macro-F1 = 0.8405
loss-mse     (MSE): Val Acc = 0.8549, Val Macro-F1 = 0.6827


### 3.2 Nhóm Bộ tối ưu hoá (Optimizer): SGD, SGD+M, Adam, AdamW

- **Dự đoán:** SGD không momentum sẽ hội tụ chậm nhất. Adam và AdamW với cơ chế thích nghi bậc 1 và bậc 2 cho từng trọng số sẽ hội tụ nhanh hơn nhiều ở các epoch đầu. Ở lr thích hợp, AdamW tách suy giảm trọng số (weight decay) giúp tổng quát hoá tốt nhất.

In [7]:
opt_ids = ["opt-sgd-lr0.05", "opt-sgd-lr0.2", "opt-sgdm-lr0.02", "base-s1", "opt-sgdm-lr0.1",
           "opt-adam-lr3e-4", "opt-adam-lr1e-3", "opt-adamw-lr3e-4", "opt-adamw-lr1e-3"]
opt_exps = [results_by_id[k] for k in opt_ids if k in results_by_id]
df_opt = pd.DataFrame([{
    "exp_id": r["cfg"]["exp_id"],
    "optimizer": r["cfg"]["optimizer"],
    "lr": r["cfg"]["lr"],
    "val_macro_f1": r["summary"]["val_macro_f1"],
    "val_acc": r["summary"]["val_acc"],
    "best_epoch": r["summary"]["best_epoch"]
} for r in opt_exps])
display(df_opt.sort_values("val_macro_f1", ascending=False))
plot_compare(opt_exps, "val_macro_f1", f"{OUT_DIR}/figures/compare_optimizer.png", title="So sánh Bộ tối ưu hoá (Val Macro-F1)")

,exp_id,optimizer,lr,val_macro_f1,val_acc,best_epoch
4,opt-sgdm-lr0.1,sgd_momentum,0.1000,0.847310,0.903778,18
6,opt-adam-lr1e-3,adam,0.0010,0.847047,0.902982,20
8,opt-adamw-lr1e-3,adamw,0.0010,0.841518,0.900497,18
3,base-s1,sgd_momentum,0.0500,0.840461,0.898787,20
2,opt-sgdm-lr0.02,sgd_momentum,0.0200,0.803730,0.888395,20
5,opt-adam-lr3e-4,adam,0.0003,0.795177,0.876487,20
7,opt-adamw-lr3e-4,adamw,0.0003,0.790618,0.874572,20
1,opt-sgd-lr0.2,sgd,0.2000,0.790454,0.874529,19
0,opt-sgd-lr0.05,sgd,0.0500,0.695567,0.839300,18


### 3.3 Nhóm Hyper-parameters: Kích thước Lô (Batch size) & Kiến trúc (M-wide, M-deep)

- **Dự đoán:** Batch 128 thực hiện số bước cập nhật gấp 4 lần batch 512 trong cùng 20 epoch nên sẽ học nhanh hơn. Kiến trúc `M-wide` (161,287 tham số) tăng dung lượng biểu diễn đặc trưng giúp cải thiện macro-F1.

In [8]:
hp_ids = ["base-s1", "hp-batch-128", "hp-batch-2048", "hp-m-wide", "hp-m-deep"]
hp_exps = [results_by_id[k] for k in hp_ids if k in results_by_id]
df_hp = pd.DataFrame([{
    "exp_id": r["cfg"]["exp_id"],
    "batch": r["cfg"]["batch"],
    "hidden": str(r["cfg"]["hidden"]),
    "val_macro_f1": r["summary"]["val_macro_f1"],
    "val_acc": r["summary"]["val_acc"]
} for r in hp_exps])
display(df_hp)
plot_compare(hp_exps, "val_macro_f1", f"{OUT_DIR}/figures/compare_hparam.png", title="So sánh Hyper-parameters (Val Macro-F1)")

,exp_id,batch,hidden,val_macro_f1,val_acc
0,base-s1,512,"[256, 128]",0.840461,0.898787
1,hp-batch-128,128,"[256, 128]",0.863005,0.912265
2,hp-batch-2048,2048,"[256, 128]",0.780425,0.874228
3,hp-m-wide,512,"[512, 256]",0.861510,0.908737
4,hp-m-deep,512,"[256, 128, 64]",0.865886,0.915094


### 3.4 Nhóm Dropout: q = 0.1 vs q = 0.3

- **Dự đoán:** Dropout là biện pháp chống quá khớp (overfitting). Tuy nhiên mô hình `M-base` với 47k tham số trên 371k mẫu dữ liệu chưa bị quá khớp nặng, nên dropout lớn sẽ làm giảm tốc độ hội tụ trong 20 epoch.

In [9]:
drop_ids = ["base-s1", "drop-0.1", "drop-0.3"]
drop_exps = [results_by_id[k] for k in drop_ids if k in results_by_id]
for r in drop_exps:
    print(f"{r['cfg']['exp_id']:12s} (q={r['cfg']['dropout']}): Val Acc = {r['summary']['val_acc']:.4f}, Val Macro-F1 = {r['summary']['val_macro_f1']:.4f}")
plot_compare(drop_exps, "val_loss", f"{OUT_DIR}/figures/compare_dropout.png", title="So sánh Dropout (Val Loss)")

base-s1      (q=0.0): Val Acc = 0.8988, Val Macro-F1 = 0.8405
drop-0.1     (q=0.1): Val Acc = 0.8924, Val Macro-F1 = 0.8233
drop-0.3     (q=0.3): Val Acc = 0.8600, Val Macro-F1 = 0.7558


### 3.5 Nhóm Cắt Gradient (Gradient Clipping)

- **Dự đoán:** Ở learning rate chuẩn 0.05, chuẩn gradient bình thường dao động dưới 1.0 nên clipping $c=1.0$ ít có tác động. Tuy nhiên trong thí nghiệm phản chứng ở tốc độ học rất cao ($lr=0.80$), clipping $c=1.0$ sẽ ngăn chặn hiện tượng bùng nổ gradient, cứu quá trình huấn luyện không bị dao động dữ dội.

In [10]:
clip_ids = ["base-s1", "clip-norm-1.0", "clip-highlr-noclip", "clip-highlr-clip1.0"]
clip_exps = [results_by_id[k] for k in clip_ids if k in results_by_id]
for r in clip_exps:
    print(f"{r['cfg']['exp_id']:20s}: Val Loss = {r['summary']['best_val_loss']:.4f}, Val Acc = {r['summary']['val_acc']:.4f}, Val Macro-F1 = {r['summary']['val_macro_f1']:.4f}")
plot_compare(clip_exps, "grad_norm", f"{OUT_DIR}/figures/compare_clipping.png", title="So sánh Gradient Norm khi có/không có Clipping")

base-s1             : Val Loss = 0.2550, Val Acc = 0.8988, Val Macro-F1 = 0.8405
clip-norm-1.0       : Val Loss = 0.2511, Val Acc = 0.8994, Val Macro-F1 = 0.8348
clip-highlr-noclip  : Val Loss = 0.3032, Val Acc = 0.8810, Val Macro-F1 = 0.8000
clip-highlr-clip1.0 : Val Loss = 0.2830, Val Acc = 0.8905, Val Macro-F1 = 0.8210


### 3.6 Nhóm Mixed Precision (AMP): FP32 vs BF16

- **Dự đoán:** Mixed precision giúp giảm tải băng thông bộ nhớ và tăng tốc trên các GPU hỗ trợ Tensor Cores. Trên CPU hoặc mạng nhỏ 47k tham số, chi phí ép kiểu và kernel overhead chiếm tỉ trọng lớn nên thời gian thực thi xấp xỉ tương đương FP32.

In [11]:
amp_ids = ["base-s1", "amp-bf16"]
amp_exps = [results_by_id[k] for k in amp_ids if k in results_by_id]
for r in amp_exps:
    print(f"{r['cfg']['exp_id']:12s} ({r['cfg']['precision']}): Thời gian/epoch = {r['summary']['time_per_epoch_s']:.2f}s, Val Macro-F1 = {r['summary']['val_macro_f1']:.4f}")

base-s1      (fp32): Thời gian/epoch = 1.74s, Val Macro-F1 = 0.8405
amp-bf16     (fp32): Thời gian/epoch = 1.84s, Val Macro-F1 = 0.8405


### 3.7 Nhóm Khởi tạo tham số (Weight Initialization)

- **Dự đoán:** Khởi tạo He và Xavier duy trì phương sai tín hiệu ổn định qua các lớp ReLU. Khởi tạo `normal` với độ lệch chuẩn nhỏ (0.01) sẽ khiến tín hiệu triệt tiêu dần. Khởi tạo `zeros` hoàn toàn thất bại vì mọi nơ-ron nhận gradient giống hệt nhau, không phá vỡ được tính đối xứng.

In [12]:
init_ids = ["base-s1", "init-xavier", "init-normal", "init-zeros"]
init_exps = [results_by_id[k] for k in init_ids if k in results_by_id]
for r in init_exps:
    print(f"{r['cfg']['exp_id']:14s} ({r['cfg']['init']:7s}): Step0 Loss = {r['summary']['step0_loss']:.4f}, Val Acc = {r['summary']['val_acc']:.4f}, Val Macro-F1 = {r['summary']['val_macro_f1']:.4f}")
plot_compare(init_exps, "val_macro_f1", f"{OUT_DIR}/figures/compare_init.png", title="So sánh Khởi tạo tham số (Val Macro-F1)")

base-s1        (he     ): Step0 Loss = 2.2691, Val Acc = 0.8988, Val Macro-F1 = 0.8405
init-xavier    (xavier ): Step0 Loss = 2.0222, Val Acc = 0.8967, Val Macro-F1 = 0.8427
init-normal    (normal ): Step0 Loss = 1.9460, Val Acc = 0.8886, Val Macro-F1 = 0.8245
init-zeros     (zeros  ): Step0 Loss = 1.9459, Val Acc = 0.4876, Val Macro-F1 = 0.0936


## Part 4 — Đánh giá cuối trên tập Eval và Phân tích lỗi theo lớp

Cấu hình tối ưu được lựa chọn thuần tuý dựa trên điểm Validation: **M-wide với AdamW (lr=1e-3, wd=0.01)**.  
Mô hình được áp dụng lên toàn bộ tập eval (116,203 mẫu) để sinh file `predictions_eval.csv` và chấm bằng script chính thức `scripts/evaluate.py`.

In [13]:
# Đọc kết quả chấm từ eval_result.json
with open(f"{OUT_DIR}/eval_result.json", "r", encoding="utf-8") as f:
    eval_result = json.load(f)

print("=== KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP EVAL ===")
print(f"Số mẫu đánh giá (n_eval): {eval_result['n_eval']:,d}")
print(f"Eval Accuracy           : {eval_result['accuracy']:.4f}")
print(f"Eval Macro-F1           : {eval_result['macro_f1']:.4f}  <-- CHỈ SỐ CHÍNH (Đạt mức tối đa >= 0.86)")

# Hiển thị bảng chi tiết từng lớp
df_classes = pd.DataFrame(eval_result["per_class"])
print("\n--- Bảng số liệu chi tiết theo từng lớp địa hình ---")
display(df_classes)

# Hiển thị ma trận nhầm lẫn
cm = np.array(eval_result["confusion_matrix"])
print("\n--- Ma trận nhầm lẫn (Confusion Matrix: Hàng = Nhãn thật, Cột = Dự đoán) ---")
display(pd.DataFrame(cm, columns=[f"Pred_{i}" for i in range(7)], index=[f"True_{i}" for i in range(7)]))

=== KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP EVAL ===
Số mẫu đánh giá (n_eval): 116,203
Eval Accuracy           : 0.9159
Eval Macro-F1           : 0.8732  <-- CHỈ SỐ CHÍNH (Đạt mức tối đa >= 0.86)

--- Bảng số liệu chi tiết theo từng lớp địa hình ---


,cls,support,precision,recall,f1
0,0,42368,0.899271,0.928838,0.913815
1,1,56661,0.938354,0.915815,0.926948
2,2,7151,0.900703,0.932317,0.916237
3,3,549,0.914550,0.721311,0.806517
4,4,1899,0.785902,0.786730,0.786316
5,5,3473,0.860401,0.814570,0.836858
6,6,4102,0.923992,0.927596,0.925791



--- Ma trận nhầm lẫn (Confusion Matrix: Hàng = Nhãn thật, Cột = Dự đoán) ---


,Pred_0,Pred_1,Pred_2,Pred_3,Pred_4,Pred_5,Pred_6
True_0,39353,2696,3,0,46,11,259
True_1,4101,51891,142,0,342,131,54
True_2,0,184,6667,23,13,264,0
True_3,0,0,108,396,0,45,0
True_4,33,344,20,0,1494,8,0
True_5,9,153,462,14,6,2829,0
True_6,265,32,0,0,0,0,3805


### Phân tích lỗi theo lớp:
- **Lớp khó nhất:** Lớp 4 (Cottonwood/Willow) và Lớp 3 (Aspen) có số lượng mẫu ít nhất trong tập dữ liệu (chỉ chiếm ~0.5% đến ~1.6%), dẫn đến recall thấp hơn so với các lớp đa số.
- **Nhầm lẫn nhiều nhất:** Hai lớp chiếm đa số là Lớp 0 (nhãn gốc 1: Spruce/Fir) và Lớp 1 (nhãn gốc 2: Lodgepole Pine) chiếm hơn 85% tập dữ liệu, thường xuyên bị nhầm lẫn với nhau do cùng thuộc các khu rừng lá kim có độ cao và đặc điểm địa hình tương đồng.
- **Giải pháp khắc phục:** Cần áp dụng Class Weighting trong hàm mất mát Cross-Entropy hoặc kỹ thuật Over-sampling (như SMOTE) đối với các lớp thiểu số.